# 02 mlp training

**Owner:** Denis. Original notebook cells: 37, 39, 40, 41, 42.

Run in the shared Databricks session using `run_pipeline.ipynb` or the documented stage runner. This is not a standalone clean-kernel notebook. Saved outputs have been removed for public-repository safety. See `docs/execution_order.md`.

### Neural Network - MLP Training & Evaluation

In [ ]:
import os
import json
import joblib
import numpy as np
import pandas as pd
import tensorflow as tf

# Reproducibility
SEED = 42
tf.keras.utils.set_random_seed(SEED)

# Shared Databricks paths
volume = "/Volumes/workspace/default/eye_style_team2"
clean_path = f"{volume}/clean/cleaned_appointments_features.csv"
baseline_dir = f"{volume}/models/baseline"

# Load saved Logistic Regression pipeline
baseline_model = joblib.load(
    f"{baseline_dir}/logistic_regression.joblib"
)

# Load original split assignments
splits = pd.read_csv(
    f"{baseline_dir}/dataset_splits.csv"
)

# Load model metadata
with open(f"{baseline_dir}/baseline_metadata.json") as f:
    metadata = json.load(f)

# Load cleaned appointments
data = pd.read_csv(clean_path)

# Recover exactly the same train/validation/test split
data = data.merge(
    splits[["AppointmentID", "Split"]],
    on="AppointmentID",
    how="left",
    validate="one_to_one"
)

assert data["Split"].notna().all()

# Use the same predictors as Logistic Regression
features = metadata["features"]
target = "target_no_show"

train = data[data["Split"] == "train"]
val = data[data["Split"] == "validation"]
test = data[data["Split"] == "test"]

# Use the previously fitted preprocessing pipeline
preprocessor = baseline_model.named_steps["preprocessor"]

def prepare_features(frame):
    transformed = preprocessor.transform(frame[features])

    if hasattr(transformed, "toarray"):
        transformed = transformed.toarray()

    return np.asarray(transformed, dtype=np.float32)

X_train_nn = prepare_features(train)
X_val_nn = prepare_features(val)

y_train_nn = train[target].to_numpy(dtype=np.float32)
y_val_nn = val[target].to_numpy(dtype=np.float32)

# Preserve the untouched test partition
print("Training:", X_train_nn.shape)
print("Validation:", X_val_nn.shape)
print("Test appointments reserved:", len(test))
print("Input dimensions:", X_train_nn.shape[1])
print("No patient overlap:",
      set(train["PatientId"]).isdisjoint(val["PatientId"])
      and set(train["PatientId"]).isdisjoint(test["PatientId"])
      and set(val["PatientId"]).isdisjoint(test["PatientId"]))

In [ ]:
from sklearn.utils.class_weight import compute_class_weight

# Define the 64 -> 32 neural-network architecture
mlp_model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(X_train_nn.shape[1],)),

    tf.keras.layers.Dense(64, activation="relu"),
    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Dense(32, activation="relu"),
    tf.keras.layers.Dropout(0.20),

    tf.keras.layers.Dense(1, activation="sigmoid")
], name="eye_style_no_show_mlp")

# Compile model
mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=[
        tf.keras.metrics.BinaryAccuracy(name="accuracy"),
        tf.keras.metrics.AUC(name="roc_auc"),
        tf.keras.metrics.AUC(
            name="pr_auc",
            curve="PR"
        )
    ]
)

mlp_model.summary()

# Handle class imbalance using training data only
classes = np.array([0, 1])

weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train_nn.astype(int)
)

class_weights = {
    int(cls): float(weight)
    for cls, weight in zip(classes, weights)
}

print("Class weights:", class_weights)

# Stop if validation loss stops improving
early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

# Train the neural network
history = mlp_model.fit(
    X_train_nn,
    y_train_nn,
    validation_data=(X_val_nn, y_val_nn),
    epochs=100,
    batch_size=64,
    class_weight=class_weights,
    callbacks=[early_stopping],
    verbose=1
)

print("\nTraining completed.")
print("Epochs completed:", len(history.history["loss"]))

In [ ]:
import matplotlib.pyplot as plt

history_df = pd.DataFrame(history.history)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Training and validation loss
axes[0].plot(history_df["loss"], label="Training")
axes[0].plot(history_df["val_loss"], label="Validation")
axes[0].set_title("MLP Training and Validation Loss")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Binary Cross-Entropy")
axes[0].legend()

# Training and validation PR-AUC
axes[1].plot(history_df["pr_auc"], label="Training")
axes[1].plot(history_df["val_pr_auc"], label="Validation")
axes[1].set_title("MLP Training and Validation PR-AUC")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("PR-AUC")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
    confusion_matrix,
    classification_report,
    ConfusionMatrixDisplay
)
import matplotlib.pyplot as plt

# Predict using restored best MLP weights
mlp_val_probs = mlp_model.predict(
    X_val_nn, batch_size=256, verbose=0
).ravel()

# Compare probability thresholds using validation data
threshold_results = []

for threshold in np.round(np.arange(0.20, 0.81, 0.05), 2):
    preds = (mlp_val_probs >= threshold).astype(int)

    threshold_results.append({
        "Threshold": float(threshold),
        "Accuracy": accuracy_score(y_val_nn, preds),
        "Precision": precision_score(
            y_val_nn, preds, zero_division=0
        ),
        "Recall": recall_score(
            y_val_nn, preds, zero_division=0
        ),
        "F1": f1_score(
            y_val_nn, preds, zero_division=0
        )
    })

mlp_threshold_df = pd.DataFrame(threshold_results)

print("=== MLP THRESHOLD COMPARISON ===")
display(mlp_threshold_df.round(4))

# Choose threshold based only on validation F1
best_mlp_row = mlp_threshold_df.loc[
    mlp_threshold_df["F1"].idxmax()
]

best_mlp_threshold = float(best_mlp_row["Threshold"])

mlp_val_preds = (
    mlp_val_probs >= best_mlp_threshold
).astype(int)

# Full validation metrics
mlp_metrics = {
    "Accuracy": accuracy_score(y_val_nn, mlp_val_preds),
    "Precision": precision_score(
        y_val_nn, mlp_val_preds, zero_division=0
    ),
    "Recall": recall_score(
        y_val_nn, mlp_val_preds, zero_division=0
    ),
    "F1": f1_score(
        y_val_nn, mlp_val_preds, zero_division=0
    ),
    "ROC-AUC": roc_auc_score(y_val_nn, mlp_val_probs),
    "PR-AUC": average_precision_score(
        y_val_nn, mlp_val_probs
    )
}

print(f"\nBest MLP threshold: {best_mlp_threshold:.2f}")

print("\n=== MLP VALIDATION METRICS ===")
for name, value in mlp_metrics.items():
    print(f"{name}: {value:.4f}")

# Compare with Logistic Regression

import json

# Recover saved baseline validation metrics
metadata_path = (
    "/Volumes/workspace/default/eye_style_team2/"
    "models/baseline/baseline_metadata.json"
)

with open(metadata_path, "r") as f:
    baseline_metadata = json.load(f)

baseline_results = baseline_metadata["validation_metrics"]

print("Baseline metrics loaded successfully.")

comparison_df = pd.DataFrame([
    {"Model": "Logistic Regression", **baseline_results},
    {"Model": "MLP (64-32)", **mlp_metrics}
]).set_index("Model")

print("\n=== MODEL COMPARISON ===")
display(comparison_df.round(4))

print("\n=== MLP CLASSIFICATION REPORT ===")
print(classification_report(
    y_val_nn,
    mlp_val_preds,
    target_names=["Attended", "No-show"],
    zero_division=0
))

# Confusion matrix
cm = confusion_matrix(y_val_nn, mlp_val_preds)

ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=["Attended", "No-show"]
).plot(cmap="Blues", values_format="d")

plt.title("MLP Validation Confusion Matrix")
plt.show()
